In [2]:
from itertools import combinations, permutations, product
from math import comb

def memoise(func):
    cache = {}

    def inner(*args):
        if args not in cache:
            cache[args] = func(*args)
        return cache[args]

    return inner

In [3]:
def det(A):
    n = len(A)
    if n == 1:
        return A[0][0]
    total = 0
    for j in range(n):
        if A[0][j] == 0:
            continue
        smaller = tuple(
            tuple(A[i][k] for k in range(n) if k != j)
            for i in range(1, n)
        )

        total += (-1)**j * A[0][j] * det(smaller)
    return total

A = ((0, 1), (1, 0))
B = ((0, 1), (0, 1))

print(det(A))
print(det(B))

-1
0


In [4]:
def matrix(board, n):
    return tuple(
        board[i*n:(i+1)*n]
        for i in range(n)
    )

def put(board, i, value):
    b = list(board)
    b[i] = value
    return tuple(b)

@memoise
def i_win(board, n, i_first=True):
    empty = [
        i for i in range(n*n)
        if board[i] == -1
    ]

    if len(empty) == 0:
        return det(matrix(board, n)) == 0

    moves = n*n - len(empty)
    my_turn = (moves % 2 == 0) == i_first

    if my_turn:
        return any(
            i_win(put(board, i, 0), n, i_first)
            for i in empty
        )

    return all(
        i_win(put(board, i, 1), n, i_first)
        for i in empty
    )

In [5]:
def winner(n, i_first=True):
    board = (-1,) * (n*n)

    if i_win(board, n, i_first):
        return "I"
        
    return "Opp"

for n in range(1, 4):
    print(
        f"n={n}: I first -> {winner(n, True)}, "
        f"Opp first -> {winner(n, False)}"
    )

n=1: I first -> I, Opp first -> Opp
n=2: I first -> I, Opp first -> I
n=3: I first -> I, Opp first -> I


In [6]:
def count_matrices(n, i_first=True):
    cells = n*n

    if i_first:
        ones = cells // 2
    else:
        ones = (cells + 1) // 2

    singular = 0

    for places in combinations(range(cells), ones):
        b = [0] * cells
        for i in places:
            b[i] = 1
        A = matrix(tuple(b), n)
        if det(A) == 0:
            singular += 1
            
    nonsingular = comb(cells, ones) - singular
    return singular, nonsingular


for n in (2, 3):
    for start in (True, False):
        first = "I" if start else "Opp"
        print(
            f"n={n}, {first} first: "
            f"{count_matrices(n, start)}"
        )

n=2, I first: (4, 2)
n=2, Opp first: (4, 2)
n=3, I first: (90, 36)
n=3, Opp first: (54, 72)


In [7]:
def det_check(A):
    n = len(A)
    answer = 0

    for p in permutations(range(n)):
        inversions = sum(
            p[i] > p[j]
            for i in range(n)
            for j in range(i+1, n)
        )

        term = (-1)**inversions
        for i in range(n):
            term *= A[i][p[i]]
        
        answer += term
    
    return answer

for n in (1, 2, 3):
    for board in product((0, 1), repeat=n*n):
        A = matrix(board, n)
        assert det(A) == det_check(A)

print("Determinants checked for n=1,2,3")

Determinants checked for n=1,2,3


In [8]:
empty = (-1,) * 9
B = put(empty, 0, 0)
print("I move first:")

for row, col in [(1, 2), (2, 1), (2, 2)]:
    i = (row-1)*3 + col-1
    result = i_win(
        put(B, i, 1), 3, True
    )
    print((row, col), result)
    assert result

I move first:
(1, 2) True
(2, 1) True
(2, 2) True


In [9]:
B = put(empty, 0, 1)
B = put(B, 4, 0)

print("Opp moves first:")

for i in range(9):
    if B[i] == -1:
        result = i_win(
            put(B, i, 1), 3, False
        )
        row = i//3 + 1
        col = i%3 + 1
        print((row, col), result)
        assert result

Opp moves first:
(1, 2) True
(1, 3) True
(2, 1) True
(2, 3) True
(3, 1) True
(3, 2) True
(3, 3) True


In [10]:
def sample_game(n, i_first=True):
    board = (-1,) * (n*n)

    for k in range(n*n):
        my_turn = (k % 2 == 0) == i_first
        value = 0 if my_turn else 1
        options = [
            put(board, i, value)
            for i in range(n*n)
            if board[i] == -1
        ]
        board = options[0]

        for next_board in options:
            result = i_win(next_board, n, i_first)
            if result == my_turn:
                board = next_board
                break

    return matrix(board, n)


A = sample_game(3, True)
for row in A:
    print(row)

print("Determinant =", det(A))

(0, 1, 0)
(1, 1, 0)
(1, 0, 0)
Determinant = 0
